# Mesurer une distance focale

**Physique-chimie · PCSI / PSI**

**Objectif** — Estimer l’incertitude-type sur la distance focale d’une lentille et comparer deux méthodes d’estimation.

**Capacité numérique du programme** — Simuler un processus aléatoire pour caractériser la variabilité d’une grandeur composée à l’aide d’un langage de programmation. Ici, la grandeur composée est la focale $f'$, calculée à partir de deux positions mesurées.

Une lentille forme l’image d’un objet. Les positions algébriques $p=\overline{OA}$ et $p'=\overline{OA'}$ sont exprimées en centimètres. La relation de conjugaison de Descartes donne

$$\frac{1}{f'}=\frac{1}{p'}-\frac{1}{p}
\qquad \Longrightarrow \qquad f'=\frac{pp'}{p-p'}.$$

Pour afficher les commandes, **exécuter toutes les cellules** avec **Run → Run All Cells**. Le premier chargement des bibliothèques peut prendre quelques secondes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from html import escape
from IPython.display import display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    import piplite
    await piplite.install("ipywidgets==8.1.8")
    import ipywidgets as widgets

## Simuler les mesures par Monte-Carlo

1. Tirer $n$ valeurs de $p$ et $n$ valeurs de $p'$ selon deux lois normales indépendantes. Les moyennes sont les positions mesurées et les écarts-types sont $u(p)$ et $u(p')$.
2. Calculer une focale pour chaque paire tirée.
3. Estimer l’incertitude-type sur $f'$ par l’écart-type des $n$ focales.

**À repérer dans le code** — Les deux lignes avec `rng.normal` simulent les mesures. La ligne suivante applique la relation de conjugaison à chaque paire. Le dernier calcul donne l’écart-type des focales.

In [ ]:
def simuler_focales(p, p_image, u_p, u_image, n):
    # 1. Tirer de nouvelles positions objet et image.
    rng = np.random.default_rng()
    positions_objet = rng.normal(p, u_p, n)
    positions_image = rng.normal(p_image, u_image, n)

    # 2. Calculer une focale pour chaque paire.
    focales = positions_objet * positions_image / (positions_objet - positions_image)

    # 3. Estimer l'incertitude-type à partir des focales simulées.
    u_monte_carlo = np.std(focales, ddof=1)
    return focales, u_monte_carlo

## Calculer l’incertitude par propagation

Pour des mesures indépendantes et des incertitudes assez petites pour linéariser $f'(p,p')$ autour des valeurs mesurées, la **formule de propagation des incertitudes** s’écrit

$$u(f') \simeq
\sqrt{\left(\frac{\partial f'}{\partial p}u(p)\right)^2+
      \left(\frac{\partial f'}{\partial p'}u(p')\right)^2}
=
\frac{\sqrt{p'^4u(p)^2+p^4u(p')^2}}{(p-p')^2}.$$

Les dérivées sont évaluées aux positions mesurées. Dans la cellule suivante, `np.hypot(a, b)` calcule $\sqrt{a^2+b^2}$. Le bouton **Simuler** appelle les deux fonctions et affiche leurs résultats côte à côte.

In [ ]:
def propager_incertitudes(p, p_image, u_p, u_image):
    numerateur = np.hypot(p_image**2 * u_p, p**2 * u_image)
    return numerateur / (p - p_image)**2

## Comparer les configurations

Choisir un scénario, puis cliquer sur **Simuler**. Les deux premiers reprennent l’activité d’origine. Modifier ensuite les champs pour essayer d’autres mesures.

L’appel Python sous les champs montre les arguments transmis à la fonction Monte-Carlo. La valeur modifiée est mise en évidence. Chaque clic lance de nouveaux tirages, même si les paramètres restent identiques.

In [ ]:
scenarios = {
    "Objet virtuel · activité d’origine": (25, 20, 0.5, 0.5),
    "Objet réel · activité d’origine": (-20, 20, 0.5, 0.5),
    "Objet réel · mesures moins précises": (-20, 20, 1.5, 1.5),
}
scenario = widgets.Dropdown(options=list(scenarios), description="Scénario",
                            style={"description_width": "initial"},
                            layout=widgets.Layout(width="min(100%, 420px)"))
positions = [widgets.FloatText(description=nom, layout=widgets.Layout(width="175px"))
             for nom in ("p (cm)", "p′ (cm)", "u(p) (cm)", "u(p′) (cm)")]
tirages = widgets.BoundedIntText(value=1000, min=10, max=100000,
                                 description="Tirages", layout=widgets.Layout(width="175px"))
lancer = widgets.Button(description="Simuler", button_style="primary")
appel = widgets.HTML()
sortie = widgets.Output()
noms_arguments = ("p", "p_image", "u_p", "u_image", "n")
champs = [*positions, tirages]
valeurs_precedentes = None
numero = 0
changement_scenario = False

def actualiser_appel(_=None):
    global valeurs_precedentes
    if changement_scenario:
        return
    valeurs = tuple(champ.value for champ in champs)
    fragments = []
    for i, (nom, valeur) in enumerate(zip(noms_arguments, valeurs)):
        argument = escape(f"{nom}={valeur:g}")
        if valeurs_precedentes is not None and valeur != valeurs_precedentes[i]:
            argument = f"<mark>{argument}</mark>"
        fragments.append(argument)
    appel.value = "<p><b>Appel Monte-Carlo</b><br><code>simuler_focales(" + ", ".join(fragments) + ")</code></p>"
    valeurs_precedentes = valeurs

def choisir_scenario(_=None):
    global changement_scenario
    changement_scenario = True
    try:
        for champ, valeur in zip(positions, scenarios[scenario.value]):
            champ.value = valeur
    finally:
        changement_scenario = False
    actualiser_appel()

def lancer_simulation(_=None):
    global numero
    p, p_image, u_p, u_image, n = (champ.value for champ in champs)
    with sortie:
        sortie.clear_output(wait=True)
        if not np.isfinite([p, p_image, u_p, u_image]).all() or min(u_p, u_image) < 0:
            print("Saisir des positions finies et des incertitudes-types positives ou nulles.")
            return
        if p == p_image:
            print("La relation de conjugaison ne donne pas de focale nominale pour p = p′.")
            return

        focales, u_monte_carlo = simuler_focales(p, p_image, u_p, u_image, n)
        if not np.isfinite(focales).all():
            print("Une paire de positions a donné une focale non finie.")
            return

        ecart = p - p_image
        focale_nominale = p * p_image / ecart
        u_propagation = propager_incertitudes(p, p_image, u_p, u_image)
        numero += 1
        print(f"Simulation {numero} · {n:,} tirages".replace(",", " "))
        print(f"Focale nominale                   {focale_nominale:.4g} cm")
        print(f"Moyenne Monte-Carlo               {np.mean(focales):.4g} cm")
        print(f"Incertitude-type · Monte-Carlo    {u_monte_carlo:.4g} cm")
        print(f"Incertitude-type · propagation    {u_propagation:.4g} cm")
        if abs(ecart) <= 5 * np.hypot(u_p, u_image):
            print("Des paires peuvent approcher p = p′. Examiner la sensibilité de l’écart-type aux tirages.")

        fig, ax = plt.subplots(figsize=(7, 3.5), layout="constrained")
        ax.hist(focales, bins=35, color="#147889", edgecolor="white")
        ax.axvline(focale_nominale, color="#9f3b24", linestyle="--", label="Focale nominale")
        ax.set(xlabel="Distance focale (cm)", ylabel="Nombre de tirages")
        ax.legend()
        plt.show()
        plt.close(fig)

for champ in champs:
    champ.observe(actualiser_appel, names="value")
scenario.observe(choisir_scenario, names="value")
lancer.on_click(lancer_simulation)
choisir_scenario()
display(widgets.VBox([
    scenario,
    widgets.Box(positions, layout=widgets.Layout(display="flex", flex_flow="row wrap")),
    widgets.HBox([tirages, lancer]),
    appel,
    sortie,
]))
lancer_simulation()

## Exploiter les résultats

1. Calculer les focales nominales des deux premiers scénarios. Comparer l’étalement des histogrammes et expliquer le rôle de $p-p'$.
2. Comparer les incertitudes-types obtenues par Monte-Carlo et par propagation dans chaque scénario. Repérer le cas où les deux méthodes concordent le mieux.
3. Dans le troisième scénario, diminuer à tour de rôle $u(p)$ puis $u(p')$ de la même quantité, en rétablissant la valeur initiale entre les deux essais. Identifier la mesure à rendre plus précise en priorité.
4. Relancer plusieurs fois la simulation avec les mêmes paramètres. Observer les valeurs qui fluctuent, puis recommencer avec davantage de tirages.

*Adapté de « Mesures et Incertitudes », activité 5.*

In [ ]:
reponses = widgets.HTML("""
<ol>
<li><b>Focales et histogrammes.</b> Objet virtuel : 25 × 20 / (25 − 20) = 100 cm.
Objet réel : (−20) × 20 / (−20 − 20) = 10 cm.
La première distribution est plus étalée. Une même variation des positions
agit davantage sur le quotient lorsque |p − p′| est petit : 5 cm contre 40 cm.</li>
<li><b>Comparaison des méthodes.</b> La formule de propagation donne environ
14,8 cm pour l’objet virtuel et 0,177 cm pour l’objet réel.
Les estimations concordent mieux pour l’objet réel : les fluctuations de p − p′
sont petites devant sa valeur absolue. Pour l’objet virtuel, la distribution
est plus dissymétrique et la linéarisation moins précise.</li>
<li><b>Précision des mesures.</b> Dans le troisième scénario, |p| = |p′| = 20 cm.
Les deux coefficients de sensibilité ont donc la même valeur absolue.
Diminuer l’une ou l’autre incertitude de la même quantité produit le même
effet dans la formule de propagation, à la fluctuation Monte-Carlo près.</li>
<li><b>Nouveaux tirages.</b> La moyenne, l’écart-type simulé et l’histogramme
varient d’une simulation à l’autre. La focale nominale et l’incertitude
calculée par propagation restent identiques. Augmenter le nombre de tirages
stabilise généralement l’estimation numérique.</li>
</ol>
""")
corrige = widgets.Accordion(children=[reponses], selected_index=None)
corrige.set_title(0, "Afficher le corrigé")
display(corrige)